In [22]:
from pathlib import Path
import re
from time import perf_counter

import cv2
import numpy as np
from paddleocr import PaddleOCR
from paddlex.model import create_model
from paddlex.inference.pipelines.ocr.result import OCRResult

CARD_IMAGE_SAMPLE = "samples/trainers-snorlax.jpg"
DEBUG_DIR = "debug"
WRITE_DEBUG_IMAGES = False
PRINT_ALL_ITEMS = False
MIN_CONFIDENT_SCORE = 0.82
FAST_RECOGNIZER_MIN_SCORE = 0.9
FAST_RECOGNIZER_MODEL = "PP-OCRv5_server_rec"
# LANGUAGE = "ch"
# LANGUAGE = "korean"
LANGUAGE = "japan"

debug_path = Path(DEBUG_DIR)
debug_path.mkdir(parents=True, exist_ok=True)

_OCR_CACHE = globals().get("_OCR_CACHE", {})
_TEXT_RECOGNIZER_CACHE = globals().get("_TEXT_RECOGNIZER_CACHE", {})


def crop_name_region(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    x1 = int(w * 0.20)
    x2 = int(w * 0.68)
    y1 = int(h * 0.035)
    y2 = int(h * 0.115)
    return image[y1:y2, x1:x2]


def crop_fast_name_line(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    y2 = int(h * 0.68)
    x2 = int(w * 0.90)
    line_crop = image[:y2, :x2]
    return cv2.copyMakeBorder(
        line_crop,
        10,
        10,
        20,
        20,
        borderType=cv2.BORDER_REPLICATE,
    )


def crop_trainer_name_region(image: np.ndarray, right_ratio: float) -> np.ndarray:
    h, w = image.shape[:2]
    x2 = int(w * right_ratio)
    return image[:, :x2]


def crop_pokemon_name_region(image: np.ndarray, left_ratio: float) -> np.ndarray:
    h, w = image.shape[:2]
    x1 = int(w * left_ratio)
    return image[:, x1:]


def add_padding(image: np.ndarray, pad_ratio: float) -> np.ndarray:
    h, w = image.shape[:2]
    pad_y = int(h * pad_ratio)
    pad_x = int(w * pad_ratio)
    return cv2.copyMakeBorder(
        image,
        pad_y,
        pad_y,
        pad_x,
        pad_x,
        borderType=cv2.BORDER_REPLICATE,
    )


def upscale(image: np.ndarray, scale: float) -> np.ndarray:
    return cv2.resize(
        image,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC,
    )


def threshold_dark_text(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )
    kernel = np.ones((2, 2), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    white_bg = 255 - mask
    return cv2.cvtColor(white_bg, cv2.COLOR_GRAY2BGR)


def sharpen(image: np.ndarray) -> np.ndarray:
    kernel = np.array(
        [
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0],
        ]
    )
    return cv2.filter2D(image, -1, kernel)


def maybe_write_debug_image(name: str, image: np.ndarray) -> None:
    if WRITE_DEBUG_IMAGES:
        cv2.imwrite(str(debug_path / name), image)


def extract_ocr_items(ocr_result) -> list[dict]:
    items = []
    for page in ocr_result:
        assert isinstance(page, OCRResult)
        for text, score, box in zip(page["rec_texts"], page["rec_scores"], page["rec_boxes"]):
            items.append(
                {
                    "text": text,
                    "raw_text": text,
                    "score": float(score),
                    "box": box.tolist() if hasattr(box, "tolist") else box,
                }
            )
    return items


def cleanup_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text.strip())
    text = re.sub(r"[・·•]+$", "", text)

    trailing_mechanic_pattern = r"(?:ex|gx|vmax|vstar|v-union|lv\.?x|v)"
    if re.search(
        rf"[\u4e00-\u9fff가-힣ァ-ンー]\s*{trailing_mechanic_pattern}$",
        text,
        flags=re.IGNORECASE,
    ):
        text = re.sub(
            rf"\s*{trailing_mechanic_pattern}$",
            "",
            text,
            flags=re.IGNORECASE,
        )
    elif re.search(r"[\u4e00-\u9fff가-힣ァ-ンー]\s*[A-Za-z]$", text):
        text = re.sub(r"\s*[A-Za-z]$", "", text)

    return text.strip()


def looks_like_name_text(text: str, min_length: int = 2) -> bool:
    text = cleanup_text(text)
    if len(text) < min_length:
        return False
    if not re.search(r"[\u4e00-\u9fff가-힣ァ-ンーA-Za-z]", text):
        return False
    bad_fragments = ["进化", "从", "進化", "から", "진화", "에서", "HP", "NO"]
    return not any(fragment in text for fragment in bad_fragments)


def longest_common_prefix(texts: list[str]) -> str:
    if not texts:
        return ""

    prefix = texts[0]
    for text in texts[1:]:
        while prefix and not text.startswith(prefix):
            prefix = prefix[:-1]
        if not prefix:
            break

    return prefix.rstrip()


def choose_best_name_candidate(ocr_items: list[dict]) -> dict | None:
    candidates = []
    for item in ocr_items:
        normalized_text = cleanup_text(item["text"])
        if not looks_like_name_text(normalized_text):
            continue
        candidates.append({**item, "text": normalized_text})
    if not candidates:
        return None

    def score(item: dict):
        text = item["text"]
        han_count = len(re.findall(r"[\u4e00-\u9fff]", text))
        hangul_count = len(re.findall(r"[가-힣]", text))
        katakana_count = len(re.findall(r"[ァ-ンー]", text))
        latin_count = len(re.findall(r"[A-Za-z]", text))
        return (
            (han_count + hangul_count + katakana_count) * 10,
            -latin_count,
            len(text),
            item["score"],
        )

    return max(candidates, key=score)


def choose_best_title_candidate(ocr_items: list[dict]) -> dict | None:
    candidates = []
    for item in ocr_items:
        normalized_text = cleanup_text(item["text"])
        if not looks_like_name_text(normalized_text, min_length=3):
            continue
        candidates.append({**item, "text": normalized_text})
    if not candidates:
        return None

    return max(candidates, key=lambda item: (item["score"], len(item["text"])))


def choose_best_trainer_candidate(ocr_items: list[dict]) -> dict | None:
    candidates = []
    for item in ocr_items:
        normalized_text = cleanup_text(item["text"])
        if not looks_like_name_text(normalized_text):
            continue
        candidates.append({**item, "text": normalized_text})
    if not candidates:
        return None

    ordered_candidates = sorted(candidates, key=lambda item: item.get("crop_ratio", 1.0))
    stable_prefix = longest_common_prefix([item["text"] for item in ordered_candidates])
    best_candidate = max(
        candidates,
        key=lambda item: (item["score"], -item.get("crop_ratio", 1.0), len(item["text"])),
    )

    if looks_like_name_text(stable_prefix):
        return {**best_candidate, "text": stable_prefix}

    return best_candidate


def count_text_support(ocr_items: list[dict], text: str) -> int:
    normalized_text = cleanup_text(text)
    return sum(1 for item in ocr_items if cleanup_text(item["text"]) == normalized_text)


def is_supported_candidate(candidate: dict | None, ocr_items: list[dict], min_support: int = 2) -> bool:
    if candidate is None:
        return False
    return count_text_support(ocr_items, candidate["text"]) >= min_support


def is_confident_candidate(candidate: dict | None, min_score: float = MIN_CONFIDENT_SCORE) -> bool:
    if candidate is None:
        return False
    script_count = len(re.findall(r"[\u4e00-\u9fff가-힣ァ-ンーA-Za-z]", candidate["text"]))
    return script_count >= 3 and candidate["score"] >= min_score


def title_has_trainer_joiner(text: str, language: str) -> bool:
    joiner_by_language = {
        "japan": "の",
        "ch": "的",
        "korean": "의",
    }
    joiner = joiner_by_language.get(language)
    return joiner is not None and joiner in text


def should_run_title_ocr(
    trainer_items: list[dict],
    best_trainer_candidate: dict | None,
    pokemon_items: list[dict],
    best_pokemon_candidate: dict | None,
) -> bool:
    trainer_is_reliable = is_confident_candidate(best_trainer_candidate) or is_supported_candidate(
        best_trainer_candidate,
        trainer_items,
    )
    pokemon_is_reliable = is_confident_candidate(best_pokemon_candidate) or is_supported_candidate(
        best_pokemon_candidate,
        pokemon_items,
    )
    return not (trainer_is_reliable and pokemon_is_reliable)


def should_prefer_title_for_pokemon(
    best_title_candidate: dict | None,
    best_trainer_candidate: dict | None,
    best_pokemon_candidate: dict | None,
    pokemon_items: list[dict],
    language: str,
) -> bool:
    if best_title_candidate is None:
        return False
    if best_pokemon_candidate is None:
        return True
    if best_trainer_candidate is not None and title_has_trainer_joiner(best_title_candidate["text"], language):
        return False
    if not is_confident_candidate(best_pokemon_candidate):
        return True
    return not is_supported_candidate(best_pokemon_candidate, pokemon_items)


def should_drop_trainer_candidate(
    best_title_candidate: dict | None,
    best_trainer_candidate: dict | None,
    language: str,
) -> bool:
    if best_title_candidate is None or best_trainer_candidate is None:
        return False
    if title_has_trainer_joiner(best_title_candidate["text"], language):
        return False
    return best_title_candidate["text"].startswith(best_trainer_candidate["text"]) and len(best_title_candidate["text"]) > len(best_trainer_candidate["text"])


def build_combined_name(
    trainer_name: str | None,
    pokemon_name: str | None,
    language: str,
) -> str | None:
    if trainer_name is None:
        return pokemon_name
    if pokemon_name is None:
        return trainer_name

    joiner_by_language = {
        "japan": "の",
        "ch": "的",
        "korean": "의",
    }
    return f"{trainer_name}{joiner_by_language.get(language, ' ')}{pokemon_name}"


def get_card_ocr(language: str) -> PaddleOCR:
    card_ocr = _OCR_CACHE.get(language)
    if card_ocr is None:
        card_ocr = PaddleOCR(
            lang=language,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )
        _OCR_CACHE[language] = card_ocr
        globals()["_OCR_CACHE"] = _OCR_CACHE
    return card_ocr


def get_text_recognizer(model_name: str):
    text_recognizer = _TEXT_RECOGNIZER_CACHE.get(model_name)
    if text_recognizer is None:
        text_recognizer = create_model(model_name)
        _TEXT_RECOGNIZER_CACHE[model_name] = text_recognizer
        globals()["_TEXT_RECOGNIZER_CACHE"] = _TEXT_RECOGNIZER_CACHE
    return text_recognizer


def build_variant(image: np.ndarray, *, pad_ratio: float, scale: float, transform: str = "color") -> np.ndarray:
    variant = upscale(add_padding(image, pad_ratio), scale)
    if transform == "sharpen":
        return sharpen(variant)
    if transform == "threshold":
        return threshold_dark_text(variant)
    return variant


def run_fast_recognizer(name_crop: np.ndarray) -> tuple[dict | None, dict]:
    recognizer = get_text_recognizer(FAST_RECOGNIZER_MODEL)
    fast_crop = upscale(crop_fast_name_line(name_crop), 1.25)
    maybe_write_debug_image("variant_fast_recognizer.jpg", fast_crop)

    fast_start = perf_counter()
    results = list(recognizer.predict(fast_crop))
    elapsed = round(perf_counter() - fast_start, 3)

    rec_text = ""
    rec_score = 0.0
    raw_text = ""
    if results:
        first_result = results[0]
        raw_text = first_result["rec_text"]
        rec_text = cleanup_text(raw_text)
        rec_score = float(first_result["rec_score"])

    candidate = None
    if looks_like_name_text(rec_text):
        candidate = {
            "text": rec_text,
            "raw_text": raw_text,
            "score": rec_score,
            "box": None,
            "variant": "fast_recognizer",
        }

    diagnostics = {
        "seconds": elapsed,
        "raw_results": results,
        "candidate": candidate,
    }
    return candidate, diagnostics


def run_trainer_recognizer(name_crop: np.ndarray) -> tuple[list[dict], dict]:
    recognizer = get_text_recognizer(FAST_RECOGNIZER_MODEL)
    trainer_items = []
    trainer_timings = {}
    trainer_start = perf_counter()

    for step in TRAINER_VARIANT_STEPS:
        variant_name = step["name"]
        trainer_crop = crop_trainer_name_region(name_crop, step["right_ratio"])
        variant_image = build_variant(
            trainer_crop,
            pad_ratio=step["pad_ratio"],
            scale=step["scale"],
            transform=step["transform"],
        )
        maybe_write_debug_image(f"variant_{variant_name}.jpg", variant_image)

        variant_start = perf_counter()
        results = list(recognizer.predict(variant_image))
        trainer_timings[variant_name] = round(perf_counter() - variant_start, 3)

        if not results:
            continue

        first_result = results[0]
        raw_text = first_result["rec_text"]
        normalized_text = cleanup_text(raw_text)
        if not looks_like_name_text(normalized_text):
            continue

        trainer_items.append(
            {
                "text": normalized_text,
                "raw_text": raw_text,
                "score": float(first_result["rec_score"]),
                "box": None,
                "variant": variant_name,
                "crop_ratio": step["right_ratio"],
            }
        )

    diagnostics = {
        "seconds": round(perf_counter() - trainer_start, 3),
        "timings": trainer_timings,
        "items": trainer_items,
    }
    return trainer_items, diagnostics


def run_pokemon_recognizer(name_crop: np.ndarray) -> tuple[list[dict], dict]:
    recognizer = get_text_recognizer(FAST_RECOGNIZER_MODEL)
    pokemon_items = []
    pokemon_timings = {}
    pokemon_start = perf_counter()

    for step in POKEMON_VARIANT_STEPS:
        variant_name = step["name"]
        pokemon_crop = crop_pokemon_name_region(name_crop, step["left_ratio"])
        variant_image = build_variant(
            pokemon_crop,
            pad_ratio=step["pad_ratio"],
            scale=step["scale"],
            transform=step["transform"],
        )
        maybe_write_debug_image(f"variant_{variant_name}.jpg", variant_image)

        variant_start = perf_counter()
        results = list(recognizer.predict(variant_image))
        pokemon_timings[variant_name] = round(perf_counter() - variant_start, 3)

        if not results:
            continue

        first_result = results[0]
        raw_text = first_result["rec_text"]
        normalized_text = cleanup_text(raw_text)
        if not looks_like_name_text(normalized_text):
            continue

        pokemon_items.append(
            {
                "text": normalized_text,
                "raw_text": raw_text,
                "score": float(first_result["rec_score"]),
                "box": None,
                "variant": variant_name,
                "crop_ratio": step["left_ratio"],
            }
        )

    diagnostics = {
        "seconds": round(perf_counter() - pokemon_start, 3),
        "timings": pokemon_timings,
        "items": pokemon_items,
    }
    return pokemon_items, diagnostics


VARIANT_STEPS = [
    {"name": "color_fast", "pad_ratio": 0.05, "scale": 1.0, "transform": "color"},
    {"name": "color_balanced", "pad_ratio": 0.08, "scale": 1.1, "transform": "color"},
    {"name": "sharpened_fallback", "pad_ratio": 0.12, "scale": 1.5, "transform": "sharpen"},
    {"name": "threshold_fallback", "pad_ratio": 0.12, "scale": 1.5, "transform": "threshold"},
]

TRAINER_VARIANT_STEPS = [
    {"name": "trainer_color_tight", "right_ratio": 0.24, "pad_ratio": 0.08, "scale": 1.0, "transform": "color"},
    {"name": "trainer_color_fast", "right_ratio": 0.26, "pad_ratio": 0.08, "scale": 1.0, "transform": "color"},
    {"name": "trainer_color_balanced", "right_ratio": 0.28, "pad_ratio": 0.08, "scale": 1.25, "transform": "color"},
    {"name": "trainer_sharpened", "right_ratio": 0.30, "pad_ratio": 0.08, "scale": 1.5, "transform": "sharpen"},
]

POKEMON_VARIANT_STEPS = [
    {"name": "pokemon_color_fast", "left_ratio": 0.30, "pad_ratio": 0.08, "scale": 1.0, "transform": "color"},
    {"name": "pokemon_color_balanced", "left_ratio": 0.30, "pad_ratio": 0.08, "scale": 1.25, "transform": "color"},
    {"name": "pokemon_color_large", "left_ratio": 0.30, "pad_ratio": 0.08, "scale": 1.75, "transform": "color"},
    {"name": "pokemon_sharpened_large", "left_ratio": 0.30, "pad_ratio": 0.08, "scale": 1.75, "transform": "sharpen"},
    {"name": "pokemon_sharpened_xl", "left_ratio": 0.30, "pad_ratio": 0.08, "scale": 2.0, "transform": "sharpen"},
]

start_total = perf_counter()

card_image_path = Path(CARD_IMAGE_SAMPLE)
assert card_image_path.exists()

card_image = cv2.imread(str(card_image_path))
assert card_image is not None

name_crop = crop_name_region(card_image)
maybe_write_debug_image("01_name_crop.jpg", name_crop)

timings = {}
fast_candidate, fast_diagnostics = run_fast_recognizer(name_crop)
timings["fast_recognizer"] = fast_diagnostics["seconds"]

best_title_candidate = None
best_trainer_candidate = None
best_pokemon_candidate = fast_candidate
title_items = []
pokemon_items = [fast_candidate] if fast_candidate is not None else []
variant_timings = {}
trainer_diagnostics = {"seconds": 0.0, "timings": {}, "items": []}
pokemon_diagnostics = {"seconds": 0.0, "timings": {}, "items": []}
used_fallback = not is_confident_candidate(best_pokemon_candidate, FAST_RECOGNIZER_MIN_SCORE)
used_title_ocr_fallback = False

if used_fallback:
    trainer_items, trainer_diagnostics = run_trainer_recognizer(name_crop)
    pokemon_region_items, pokemon_diagnostics = run_pokemon_recognizer(name_crop)
    pokemon_items.extend(pokemon_region_items)

    best_trainer_candidate = choose_best_trainer_candidate(trainer_items)
    best_pokemon_candidate = choose_best_name_candidate(pokemon_region_items) or choose_best_name_candidate(pokemon_items)

    if should_run_title_ocr(
        trainer_items,
        best_trainer_candidate,
        pokemon_items,
        best_pokemon_candidate,
    ):
        used_title_ocr_fallback = True
        card_ocr = get_card_ocr(LANGUAGE)

        for step in VARIANT_STEPS:
            variant_name = step["name"]
            variant_image = build_variant(
                name_crop,
                pad_ratio=step["pad_ratio"],
                scale=step["scale"],
                transform=step["transform"],
            )
            maybe_write_debug_image(f"variant_{variant_name}.jpg", variant_image)

            variant_start = perf_counter()
            result = card_ocr.predict(
                variant_image,
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False,
            )
            variant_timings[variant_name] = round(perf_counter() - variant_start, 3)

            items = extract_ocr_items(result)
            for item in items:
                item["variant"] = variant_name

            title_items.extend(items)
            best_title_candidate = choose_best_title_candidate(title_items)

            if best_title_candidate is not None and best_title_candidate["score"] >= MIN_CONFIDENT_SCORE:
                break

if should_prefer_title_for_pokemon(
    best_title_candidate,
    best_trainer_candidate,
    best_pokemon_candidate,
    pokemon_items,
    LANGUAGE,
):
    best_pokemon_candidate = {**best_title_candidate}

if should_drop_trainer_candidate(best_title_candidate, best_trainer_candidate, LANGUAGE):
    best_trainer_candidate = None

trainer_name = best_trainer_candidate["text"] if best_trainer_candidate is not None else None
pokemon_name = best_pokemon_candidate["text"] if best_pokemon_candidate is not None else None
combined_name = (
    best_title_candidate["text"]
    if best_title_candidate is not None
    else build_combined_name(trainer_name, pokemon_name, LANGUAGE)
)

extracted_names = {
    "trainer_name": trainer_name,
    "pokemon_name": pokemon_name,
    "combined_name": combined_name,
}

print("Fast path timings (seconds):")
print(timings)
print("Used split fallback:", used_fallback)
print("Used title OCR fallback:", used_title_ocr_fallback)

if used_title_ocr_fallback:
    print("Title OCR timings (seconds):")
    print(variant_timings)

if used_fallback:
    print("Trainer OCR timings (seconds):")
    print(trainer_diagnostics["timings"])
    print("Pokemon OCR timings (seconds):")
    print(pokemon_diagnostics["timings"])

if PRINT_ALL_ITEMS:
    print()
    print("Fast recognizer raw results:")
    print(fast_diagnostics["raw_results"])
    print()
    print("Title OCR items:")
    for item in title_items:
        print(item)
    print()
    print("Trainer OCR items:")
    for item in trainer_diagnostics["items"]:
        print(item)
    print()
    print("Pokemon OCR items:")
    for item in pokemon_diagnostics["items"]:
        print(item)

print()
print("Extracted names:")
print(extracted_names)
print()
print("Best full title candidate:")
print(best_title_candidate)
print()
print("Best trainer candidate:")
print(best_trainer_candidate)
print()
print("Best pokemon candidate:")
print(best_pokemon_candidate)
print()
print(f"Total seconds: {perf_counter() - start_total:.3f}")

Fast path timings (seconds):
{'fast_recognizer': 0.091}
Used split fallback: True
Used title OCR fallback: False
Trainer OCR timings (seconds):
{'trainer_color_tight': 0.064, 'trainer_color_fast': 0.065, 'trainer_color_balanced': 0.065, 'trainer_sharpened': 0.065}
Pokemon OCR timings (seconds):
{'pokemon_color_fast': 0.065, 'pokemon_color_balanced': 0.065, 'pokemon_color_large': 0.064, 'pokemon_sharpened_large': 0.064, 'pokemon_sharpened_xl': 0.064}

Extracted names:
{'trainer_name': 'ホップ', 'pokemon_name': 'カビゴン', 'combined_name': 'ホップのカビゴン'}

Best full title candidate:
None

Best trainer candidate:
{'text': 'ホップ', 'raw_text': 'ホップの', 'score': 0.8338896036148071, 'box': None, 'variant': 'trainer_color_fast', 'crop_ratio': 0.26}

Best pokemon candidate:
{'text': 'カビゴン', 'raw_text': 'カビゴン', 'score': 0.7678800821304321, 'box': None, 'variant': 'pokemon_sharpened_xl', 'crop_ratio': 0.3}

Total seconds: 0.705
